# Evaluate U-Net performance over time

This notebook is the home for multi-date model evaluation. These routines intentionally remain notebook analysis rather than public visualization APIs.

The example uses the demo bundle produced by `2-U-Net_Fit_Generic.ipynb`. Replace the bundle and data-loading block with matching real data when evaluating another model.

In [ ]:
from pathlib import Path
import sys

for parent in (Path.cwd(), Path.cwd().parent):
    if (parent / "mindthegap").is_dir():
        sys.path.insert(0, str(parent))
        break

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import mindthegap as mtg

model, metadata = mtg.load_model_bundle("demo-unet-bundle")
input_vars = [item["name"] for item in metadata["inputs"]]
ds, _ = mtg.demo_data(days=120, lat_size=16, lon_size=16, seed=42)
train_end = metadata["dataset"]["training_period"].split(" to ")[1]
standardized_vars = [
    name
    for name, values in metadata["preprocessing"]["standardization"].items()
    if values.get("applied")
]
ds_std, _ = mtg.build_standardized_lazy(
    ds,
    target_variable="chlor_a",
    missing_flag="cloud_flag",
    land_flag="land_flag",
    train_dates=slice(str(ds.time.values[0])[:10], train_end),
    std_vars=standardized_vars,
    log_target=True,
    missing_flag_shift=10,
    n_temporal_lags=metadata["preprocessing"]["transforms"]["temporal_lags"],
    output_chunks={"time": 20, "lat": 16, "lon": 16},
)

## Self-Supervised Evaluation

Evaluate on held-out **synthetic clouds** only (pixels intentionally hidden) and compare with a persistence baseline. This code was moved from the training notebook for later development and has not been validated.

In [ ]:
# Define test period (after training and validation)
test_start = val_end_date + pd.DateOffset(days=2)  # 2-day buffer
test_end = pd.to_datetime(ds.time.values[-1])

print(f"Test period: {test_start.date()} to {test_end.date()}")

# Select test data from original lazy dataset
ds_test = ds_std.sel(time=slice(str(test_start.date()), str(test_end.date())))

# Load test set into memory for prediction
print("Loading test data...")
ds_test_loaded = ds_test.load()

# Stack input channels
X_test = np.stack([ds_test_loaded[ch].values for ch in X_vars], axis=-1).astype('float32')
print(f"Test data shape: {X_test.shape}")

In [ ]:
# Predict on test set
print("\nPredicting on test set...")
predictions_std = model.predict(X_test, batch_size=1, verbose=1)

# Unstandardize predictions
predictions = predictions_std[..., 0] * y_std + y_mean

# Get ground truth from original data (in log-space if log_transform=True)
truth_data = ds[target_var].sel(time=slice(str(test_start.date()), str(test_end.date()))).values
if log_transform:
    truth = np.log(np.where(truth_data > 0, truth_data, np.nan))
else:
    truth = truth_data

# Get synthetic cloud mask for test period
synth_cloud_test = ds_test_loaded['estimate_flag'].values

# Compute metrics only at synthetic clouds (held-out observed pixels)
mask_valid = (synth_cloud_test == 1) & np.isfinite(truth) & np.isfinite(predictions)
test_mae = np.mean(np.abs(predictions[mask_valid] - truth[mask_valid]))
test_rmse = np.sqrt(np.mean((predictions[mask_valid] - truth[mask_valid])**2))

print(f"\nTest set pixels evaluated: {mask_valid.sum():,}")
print(f"U-Net MAE:  {test_mae:.4f}")
print(f"U-Net RMSE: {test_rmse:.4f}")

In [ ]:
# Persistence baseline (yesterday's value)
print("\nComputing persistence baseline...")
if log_transform:
    all_truth = np.log(np.where(ds[target_var].values > 0, ds[target_var].values, np.nan))
else:
    all_truth = ds[target_var].values

# Get test period indices in full dataset
test_time_idx = np.where((ds.time.values >= np.datetime64(test_start)) & 
                         (ds.time.values <= np.datetime64(test_end)))[0]
persistence = all_truth[test_time_idx - 1]  # Previous day's value

mask_persist = (synth_cloud_test == 1) & np.isfinite(truth) & np.isfinite(persistence)
persist_mae = np.mean(np.abs(persistence[mask_persist] - truth[mask_persist]))
persist_rmse = np.sqrt(np.mean((persistence[mask_persist] - truth[mask_persist])**2))

print(f"Persistence baseline:")
print(f"  MAE:  {persist_mae:.4f}")
print(f"  RMSE: {persist_rmse:.4f}")

In [ ]:
# Summary
print("\n" + "="*60)
print("SELF-SUPERVISED TEST RESULTS")
print("="*60)
print(f"Target variable: {target_var}")
print(f"Evaluation metric: {'log-space' if log_transform else 'linear'} MAE/RMSE")
print(f"Evaluated at {mask_valid.sum()} synthetic cloud pixels")
print()
print(f"{'Metric':<15} {'U-Net':>10} {'Persist':>10} {'Improve':>10}")
print("-" * 60)
print(f"{'MAE':<15} {test_mae:>10.4f} {persist_mae:>10.4f} {(persist_mae-test_mae):>10.4f}")
print(f"{'RMSE':<15} {test_rmse:>10.4f} {persist_rmse:>10.4f} {(persist_rmse-test_rmse):>10.4f}")
print(f"{'% Better':<15} {(1-test_mae/persist_mae)*100:>9.1f}% {(1-test_rmse/persist_rmse)*100:>9.1f}%")
print("="*60)

if test_mae < persist_mae:
    print("\n✓ SUCCESS: Model beats persistence baseline!")
else:
    print("\n✗ Model does not beat persistence. Consider:")
    print("  - More training epochs or data")
    print("  - Additional predictor features (SST, winds, etc.)")
    print("  - Different missing_flag_shift value")
    print("  - Spatial patching (if memory-limited)")

## Daily held-out-pixel error

The metric below evaluates only synthetic gaps: pixels whose true values are known but were intentionally hidden from the predictors.

In [ ]:
def daily_synthetic_gap_mae(dataset, model, metadata, block_size=30):
    input_vars = [item["name"] for item in metadata["inputs"]]
    target_stats = metadata["preprocessing"]["standardization"]["full_target"]
    mean, std = target_stats["mean"], target_stats["std"]
    values = []

    for start in range(0, dataset.sizes["time"], block_size):
        block = dataset.isel(time=slice(start, start + block_size)).load()
        inputs = np.stack(
            [np.nan_to_num(block[name].values, nan=0.0) for name in input_vars],
            axis=-1,
        ).astype("float32")
        prediction = model.predict(inputs, batch_size=1, verbose=0)[..., 0]
        prediction = prediction * std + mean
        truth = block["full_target"].values * std + mean
        mask = (
            (block["estimate_flag"].values == 1)
            & np.isfinite(truth)
            & np.isfinite(prediction)
        )
        error = np.where(mask, np.abs(truth - prediction), np.nan)
        values.extend(np.nanmean(error, axis=(1, 2)))

    return pd.Series(values, index=pd.to_datetime(dataset.time.values), name="MAE")

In [ ]:
daily_mae = daily_synthetic_gap_mae(ds_std, model, metadata)
ax = daily_mae.plot(figsize=(11, 4), title="Daily synthetic-gap MAE")
ax.set_ylabel("MAE in model output units")
ax.grid(alpha=0.3)
plt.show()

## Missing-data coverage

Plot missing-water coverage beside error to investigate whether performance changes with cloud amount.

In [ ]:
ocean = ds_std["land_flag"] == 0
missing = ds_std["unavailable_flag"] == 1
missing_fraction = (missing & ocean).sum(("lat", "lon")) / ocean.sum(("lat", "lon"))

fig, first = plt.subplots(figsize=(11, 4))
first.plot(daily_mae.index, daily_mae.values, color="tab:red")
first.set_ylabel("synthetic-gap MAE", color="tab:red")
second = first.twinx()
second.plot(ds_std.time.values, missing_fraction, color="tab:blue", alpha=0.7)
second.set_ylabel("missing-water fraction", color="tab:blue")
first.set_xlabel("time")
first.grid(alpha=0.3)
plt.show()